In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anonymous111111111/spam-cordinates/spam_coordinates.csv
/kaggle/input/datasets/anonymous111111111/spam-cordinates/participants.csv
/kaggle/input/datasets/anonymous111111111/responses/vft_responses_hindi_only.csv
/kaggle/input/datasets/anonymous111111111/devnagri-mapping/unique_words.txt


In [2]:
# Install once
!pip install sentence-transformers -q

mapping words written in English alphabets to their respective devnagri words

In [3]:
import pandas as pd
import re

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/anonymous111111111/responses/vft_responses_hindi_only.csv')

# -----------------------------
# Step 1: Load mapping file
# -----------------------------
mapping = {}

with open('/kaggle/input/datasets/anonymous111111111/devnagri-mapping/unique_words.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            roman = parts[0].lower()
            devanagari = parts[1]
            mapping[roman] = devanagari

# -----------------------------
# Step 2: Helper functions
# -----------------------------

# Check if word is Devanagari
def is_devanagari(word):
    return bool(re.search(r'[\u0900-\u097F]', str(word)))

# Convert word
def convert_word(word):
    if pd.isna(word):
        return word
    
    word = str(word).strip()
    
    # If already Devanagari → return as is
    if is_devanagari(word):
        return word
    
    # If English → map it
    return mapping.get(word.lower(), None)  # None if not found

# -----------------------------
# Step 3: Create new column
# -----------------------------
df['devnagri_word'] = df['word'].apply(convert_word)

# -----------------------------
# Step 4: Save new CSV
# -----------------------------
output_path = '/kaggle/working/final_normalised_dataset.csv'
df.to_csv(output_path, index=False)

print(f"Saved updated dataset to: {output_path}")

Saved updated dataset to: /kaggle/working/final_normalised_dataset.csv


applying transformation and creating new row devnagri words

In [4]:
import pandas as pd
import re

# -----------------------------
# Load datasets
# -----------------------------
main_df = pd.read_csv('/kaggle/input/datasets/anonymous111111111/spam-cordinates/spam_coordinates.csv')   # your current dataset
vft_df = pd.read_csv('/kaggle/input/datasets/anonymous111111111/responses/vft_responses_hindi_only.csv')  # reference dataset

# -----------------------------
# Step 1: Filter participants
# -----------------------------
valid_participants = set(vft_df['participant_id'])

filtered_df = main_df[main_df['participant_id'].isin(valid_participants)].copy()

print(f"After filtering: {len(filtered_df)} rows")

# -----------------------------
# Step 2: Load mapping
# -----------------------------
mapping = {}

with open('/kaggle/input/datasets/anonymous111111111/devnagri-mapping/unique_words.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            roman = parts[0].lower()
            devanagari = parts[1]
            mapping[roman] = devanagari

# -----------------------------
# Step 3: Helpers
# -----------------------------
def is_devanagari(word):
    return bool(re.search(r'[\u0900-\u097F]', str(word)))

def convert_word(word):
    if pd.isna(word):
        return word
    
    word = str(word).strip()
    
    # Already Devanagari → keep
    if is_devanagari(word):
        return word
    
    # Roman → map
    return mapping.get(word.lower(), word)  # fallback = original

# -----------------------------
# Step 4: Apply transformation
# -----------------------------
filtered_df['devnagri_word'] = filtered_df['word'].apply(convert_word)

# -----------------------------
# Step 5: Save new dataset
# -----------------------------
output_path = '/kaggle/working/final_filtered_mapped_dataset.csv'
filtered_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

After filtering: 682 rows
Saved to: /kaggle/working/final_filtered_mapped_dataset.csv


creating embeddings for hindi words using pre-trained model

In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv('/kaggle/working/final_normalised_dataset.csv')

# -----------------------------
# Step 1: Get unique words
# -----------------------------
words = df['devnagri_word'].dropna().astype(str).str.strip().unique()

print(f"Total unique words: {len(words)}")

# -----------------------------
# Step 2: Load IndicSBERT
# -----------------------------
model = SentenceTransformer('sentence-transformers/LaBSE')

# -----------------------------
# Step 3: Generate embeddings
# -----------------------------
embeddings = model.encode(words, batch_size=32, show_progress_bar=True)

# embeddings shape → (num_words, embedding_dim)
print("Embedding shape:", embeddings.shape)

# -----------------------------
# Step 4: Save embeddings
# -----------------------------

# (A) Save as numpy file (best for ML)
np.save('/kaggle/working/hindi_word_embeddings.npy', embeddings)
np.save('/kaggle/working/hindi_words.npy', words)

# (B) Save mapping (word → vector) as CSV
emb_df = pd.DataFrame(embeddings)
emb_df.insert(0, 'word', words)

# emb_df.to_csv('/mnt/data/hindi_word_embeddings.csv', index=False)

print("Saved embeddings successfully!")

Total unique words: 217


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embedding shape: (217, 768)
Saved embeddings successfully!


sanity check

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

word_to_idx = {w: i for i, w in enumerate(words)}

w1, w2 = "शेर", "बाघ"

if w1 in word_to_idx and w2 in word_to_idx:
    sim = cosine_similarity(
        [embeddings[word_to_idx[w1]]],
        [embeddings[word_to_idx[w2]]]
    )
    print(sim[0][0])

0.7289387


Hypothesis check : words that are semantically similar , such words were placed in clusters by participants 

In [7]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load dataset (ONLY ONE FILE)
# -----------------------------
df = pd.read_csv('/kaggle/working/final_filtered_mapped_dataset.csv')

# -----------------------------
# Load embeddings + words (CRITICAL)
# -----------------------------
embeddings = np.load('/kaggle/working/hindi_word_embeddings.npy')
words = np.load('/kaggle/working/hindi_words.npy', allow_pickle=True)

word_to_emb = {word: embeddings[i] for i, word in enumerate(words)}

# -----------------------------
# Function: analyze one group
# -----------------------------
def analyze_group(group):
    words = group['devnagri_word'].values
    coords = group[['x_norm', 'y_norm']].values
    
    if len(words) < 3:
        return None
    
    # Keep only words that have embeddings
    valid = [(w, c) for w, c in zip(words, coords) if w in word_to_emb]
    
    if len(valid) < 3:
        return None
    
    words, coords = zip(*valid)
    coords = np.array(coords)
    
    emb = np.array([word_to_emb[w] for w in words])
    
    # Cosine similarity → distance
    cos_sim = cosine_similarity(emb)
    semantic_dist = 1 - cos_sim
    
    # Spatial distance
    spatial_dist = squareform(pdist(coords, metric='euclidean'))
    
    # Flatten upper triangle
    idx = np.triu_indices(len(words), k=1)
    
    sem_vals = semantic_dist[idx]
    spat_vals = spatial_dist[idx]
    
    # Spearman correlation
    corr, _ = spearmanr(sem_vals, spat_vals)
    
    return corr

# -----------------------------
# Run per participant + category
# -----------------------------
results = []

for (pid, cat), group in df.groupby(['participant_id', 'domain']):
    corr = analyze_group(group)
    
    if corr is not None:
        results.append({
            'participant_id': pid,
            'category': cat,
            'correlation': corr
        })

results_df = pd.DataFrame(results)

# -----------------------------
# Aggregate by category
# -----------------------------
category_summary = results_df.groupby('category')['correlation'].mean().reset_index()

print(category_summary)

     category  correlation
0     animals     0.183937
1  body-parts     0.380866
2     colours     0.087906
3       foods     0.291428


hypothesis failed, or showed very weak association

In [8]:
print(df.columns)

Index(['participant_id', 'domain', 'word', 'x_norm', 'y_norm',
       'devnagri_word'],
      dtype='object')


In [9]:
from scipy.stats import ttest_1samp

for cat in results_df['category'].unique():
    vals = results_df[results_df['category'] == cat]['correlation']
    
    stat, pval = ttest_1samp(vals, 0)
    
    print(cat, "p-value:", pval)

animals p-value: 0.0008330205721023742
body-parts p-value: 4.987602589752753e-08
foods p-value: 6.552671468029001e-07
colours p-value: 0.40191303474146217


Checking if sematantically similar words are retrieved in clusters or not 

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Data from your analysis results ──────────────────────────────────────────
domains = ['Animals', 'Body-parts', 'Foods', 'Colours']

between = [6492, 7002, 6338, 6895]
within  = [5737, 5295, 4986, 4361]

quartiles = {
    'Animals':    [6981, 6297, 5632, 5443],
    'Body-parts': [6956, 5424, 5217, 4812],
    'Foods':      [6209, 4879, 4781, 5523],
    'Colours':    [6895, 4249, 4298, 4448],
}

spearman = {
    'Animals':    (-0.117, 0.094),
    'Body-parts': (-0.188, 0.014),
    'Foods':      (-0.070, 0.328),
    'Colours':    (-0.211, 0.210),
}

# ── Style ─────────────────────────────────────────────────────────────────────
BG      = '#FFFFFF'
CARD    = '#F5F7FA'
ACCENT1 = '#1565C0'   # blue  – within-cluster
ACCENT2 = '#C62828'   # red   – between-cluster
GOLD    = '#E65100'
TEXT    = '#1A1A2E'
MUTED   = '#555770'
SIG_COLOR = '#1565C0'
NS_COLOR  = '#BDBDBD'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.facecolor': CARD,
    'figure.facecolor': BG,
})

fig = plt.figure(figsize=(14, 5.2), facecolor=BG)
gs = fig.add_gridspec(1, 3, left=0.05, right=0.97, top=0.82, bottom=0.18,
                      wspace=0.38)

# ─────────────────────────────────────────────────────────────────────────────
# Panel A: Cluster-switch bar chart
# ─────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(CARD)
for spine in ax1.spines.values():
    spine.set_color('#CCCCCC')

x = np.arange(len(domains))
w = 0.32
ax1.bar(x - w/2, between, width=w, color=ACCENT2, alpha=0.85, label='Between-cluster', zorder=3)
ax1.bar(x + w/2, within,  width=w, color=ACCENT1, alpha=0.85, label='Within-cluster',  zorder=3)

for i in range(len(domains)):
    delta = between[i] - within[i]
    ymax  = max(between[i], within[i])
    ax1.annotate(f'−{delta}ms', xy=(x[i], ymax + 100),
                 ha='center', fontsize=7.5, color=GOLD, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(domains, fontsize=8.5, color=TEXT)
ax1.set_ylabel('Mean IRT (ms)', color=MUTED, fontsize=9)
ax1.tick_params(colors=MUTED, labelcolor=TEXT)
ax1.set_ylim(0, 8800)
ax1.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax1.set_axisbelow(True)

patch1 = mpatches.Patch(color=ACCENT2, alpha=0.85, label='Between-cluster')
patch2 = mpatches.Patch(color=ACCENT1, alpha=0.85, label='Within-cluster')
ax1.legend(handles=[patch1, patch2], fontsize=7.5, framealpha=0.7,
           labelcolor=TEXT, loc='upper right')
ax1.text(0.5, 1.07, 'A  Within vs Between Cluster IRT',
         transform=ax1.transAxes, ha='center', fontsize=10, color=TEXT, fontweight='bold')

# ─────────────────────────────────────────────────────────────────────────────
# Panel B: Quartile line plot
# ─────────────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(CARD)
for spine in ax2.spines.values():
    spine.set_color('#CCCCCC')

line_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']
markers = ['o', 's', '^', 'D']
qlabels = ['Q1\n(least)', 'Q2', 'Q3', 'Q4\n(most)']

for idx, (dom, vals) in enumerate(quartiles.items()):
    ax2.plot([1,2,3,4], vals, color=line_colors[idx], marker=markers[idx],
             linewidth=1.8, markersize=6, label=dom, zorder=3)

ax2.set_xticks([1,2,3,4])
ax2.set_xticklabels(qlabels, fontsize=8, color=TEXT)
ax2.set_ylabel('Mean IRT (ms)', color=MUTED, fontsize=9)
ax2.tick_params(colors=MUTED, labelcolor=TEXT)
ax2.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax2.set_axisbelow(True)
ax2.set_xlim(0.7, 4.3)
ax2.legend(fontsize=7.5, framealpha=0.7, labelcolor=TEXT, loc='upper right')
ax2.text(0.5, 1.07, 'B  Mean IRT by Similarity Quartile',
         transform=ax2.transAxes, ha='center', fontsize=10, color=TEXT, fontweight='bold')

# ─────────────────────────────────────────────────────────────────────────────
# Panel C: Spearman r horizontal bars
# ─────────────────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor(CARD)
for spine in ax3.spines.values():
    spine.set_color('#CCCCCC')

rs = [spearman[d][0] for d in domains]
ps = [spearman[d][1] for d in domains]
bar_colors = [SIG_COLOR if p < 0.05 else NS_COLOR for p in ps]

ypos = np.arange(len(domains))
ax3.barh(ypos, rs, color=bar_colors, height=0.45, zorder=3, edgecolor='none')

for i, (r, p) in enumerate(zip(rs, ps)):
    label = f'r={r:.3f}  p={p:.3f}' + ('  *' if p < 0.05 else '')
    ax3.text(min(r, -0.005), i, label, va='center', ha='right',
             fontsize=8, color=TEXT if p < 0.05 else MUTED)

ax3.axvline(0, color=MUTED, linewidth=0.8, zorder=2)
ax3.set_yticks(ypos)
ax3.set_yticklabels(domains, fontsize=9, color=TEXT)
ax3.tick_params(colors=MUTED, labelcolor=TEXT)
ax3.xaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax3.set_axisbelow(True)
ax3.set_xlabel('Spearman r  (cosine sim vs IRT)', color=MUTED, fontsize=8.5)
ax3.set_xlim(-0.35, 0.05)

sig_patch = mpatches.Patch(color=SIG_COLOR, label='p < 0.05')
ns_patch  = mpatches.Patch(color=NS_COLOR,  label='p ≥ 0.05')
ax3.legend(handles=[sig_patch, ns_patch], fontsize=7.5,
           framealpha=0.7, labelcolor=TEXT, loc='lower right')
ax3.text(0.5, 1.07, 'C  Spearman Correlation per Domain',
         transform=ax3.transAxes, ha='center', fontsize=10, color=TEXT, fontweight='bold')

# ── Super-title ───────────────────────────────────────────────────────────────
fig.text(0.5, 0.96,
         'H1: Semantically Similar Words Retrieved with Lower IRT',
         ha='center', fontsize=13, color=TEXT, fontweight='bold')
fig.text(0.5, 0.90,
         'LaBSE cosine similarity vs Inter-Response Time  |  n = 606 consecutive word pairs',
         ha='center', fontsize=8.5, color=MUTED)

plt.savefig('H1_poster_figure.png', dpi=200, bbox_inches='tight', facecolor=BG)
print("Saved → H1_poster_figure.png")

Saved → H1_poster_figure.png


In [11]:
# =============================================================================
# H_new: Retrieval Becomes Increasingly Semantically Clustered Over Time
# =============================================================================
# Hypothesis: Early retrievals in a VFT sequence are semantically diffuse
# (random-like), while later retrievals are semantically concentrated within
# tighter neighbourhoods — i.e. mean cosine similarity between consecutive
# word pairs increases as the sequence progresses.
#
# Tests:
#   T1 — Split each session into thirds (early / mid / late), compute mean
#          cosine similarity per third, run Friedman test + Wilcoxon
#          signed-rank (early vs late)
#   T2 — Per-session Spearman ρ between word position and consecutive-pair
#          cosine similarity, then one-sample Wilcoxon against zero to test
#          whether median ρ > 0 across sessions
#   T3 — Permutation baseline: shuffle word order within each session,
#          recompute ρ, compare observed vs shuffled distribution
#
# Visualizations:
#   Fig 1 — Mean cosine similarity per third (early/mid/late) per domain
#   Fig 2 — Distribution of session-level Spearman ρ values per domain
#   Fig 3 — Observed vs permutation null distribution of ρ
#   Fig 4 — Example session: cosine similarity vs word position (line plot)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, wilcoxon, friedmanchisquare
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── style (same as your other notebooks) ──────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'savefig.dpi':       150,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

C_EARLY  = '#C62828'   # red
C_MID    = '#E65100'   # orange
C_LATE   = '#1565C0'   # blue
C_NULL   = '#BDBDBD'
TEXT     = '#1A1A2E'
MUTED    = '#555770'
CARD     = '#F5F7FA'

DOMAIN_COLORS = {
    'animals':    '#534AB7',
    'body-parts': '#1D9E75',
    'foods':      '#D85A30',
    'colours':    '#E0A820',
}

# ── 0. Load data ──────────────────────────────────────────────────────────────
print("=" * 65)
print("H_new: RETRIEVAL BECOMES MORE SEMANTICALLY CLUSTERED OVER TIME")
print("=" * 65)

VFT_PATH = '/kaggle/working/final_normalised_dataset.csv'

vft = pd.read_csv(VFT_PATH)
vft['IRT']        = pd.to_numeric(vft['IRT'],        errors='coerce')
vft['word_order'] = pd.to_numeric(vft['word_order'], errors='coerce')
vft = vft.dropna(subset=['devnagri_word', 'word_order'])
vft = vft.sort_values(['participant_id', 'domain', 'word_order'])

DOMAINS = vft['domain'].dropna().unique().tolist()
print(f"VFT rows: {len(vft)} | Participants: {vft['participant_id'].nunique()}")
print(f"Domains: {DOMAINS}\n")

# ── 1. Generate LaBSE embeddings for all unique words ─────────────────────────
print("Loading LaBSE...")
model = SentenceTransformer('sentence-transformers/LaBSE')

unique_words = vft['devnagri_word'].dropna().unique()
print(f"Embedding {len(unique_words)} unique words...")
emb_matrix = model.encode(unique_words, batch_size=32, show_progress_bar=True)
word2emb   = {w: emb_matrix[i] for i, w in enumerate(unique_words)}
print("Embeddings ready.\n")

# ── 2. Build per-session pair records with position info ──────────────────────
# For each consecutive pair (word_i, word_{i+1}) store:
#   cosine similarity, position of word_{i+1}, session thirds label
records = []

MIN_WORDS = 6   # skip sessions too short to split into thirds

for (pid, dom), grp in vft.groupby(['participant_id', 'domain']):
    grp = grp.sort_values('word_order').reset_index(drop=True)
    words = grp['devnagri_word'].tolist()

    if len(words) < MIN_WORDS:
        continue

    n = len(words)
    third = n // 3

    for i in range(len(words) - 1):
        w0 = words[i]
        w1 = words[i + 1]
        if w0 not in word2emb or w1 not in word2emb:
            continue

        sim = float(cosine_similarity(
            word2emb[w0].reshape(1, -1),
            word2emb[w1].reshape(1, -1)
        )[0, 0])

        pos = i + 1   # position of w1 (1-indexed)

        # assign third
        if pos <= third:
            segment = 'early'
        elif pos <= 2 * third:
            segment = 'mid'
        else:
            segment = 'late'

        records.append({
            'participant_id': pid,
            'domain':         dom,
            'w0':             w0,
            'w1':             w1,
            'position':       pos,
            'cosine_sim':     sim,
            'segment':        segment,
            'n_words':        n,
        })

pairs_df = pd.DataFrame(records)
print(f"Total consecutive pairs: {len(pairs_df)}")
print(pairs_df.groupby('segment')['cosine_sim'].agg(['mean', 'count']))
print()

# ── 3. T1 — Thirds analysis: Friedman + Wilcoxon (early vs late) ──────────────
print("=" * 65)
print("T1: MEAN COSINE SIMILARITY PER THIRD (early / mid / late)")
print("=" * 65)

# Per-session mean sim per third
session_thirds = (
    pairs_df.groupby(['participant_id', 'domain', 'segment'])['cosine_sim']
    .mean()
    .reset_index()
    .rename(columns={'cosine_sim': 'mean_sim'})
)

thirds_summary = {}

for dom in DOMAINS:
    sub = session_thirds[session_thirds['domain'] == dom]
    wide = sub.pivot(index='participant_id', columns='segment', values='mean_sim').dropna()

    if wide.shape[0] < 3:
        print(f"\n[{dom}] Not enough sessions for test (n={wide.shape[0]}), skipping.")
        continue

    # ensure columns exist
    cols_present = [c for c in ['early', 'mid', 'late'] if c in wide.columns]
    if len(cols_present) < 2:
        continue

    early = wide['early'].values if 'early' in wide else np.array([])
    mid   = wide['mid'].values   if 'mid'   in wide else np.array([])
    late  = wide['late'].values  if 'late'  in wide else np.array([])

    print(f"\n[{dom}]  n sessions = {len(wide)}")
    print(f"  Mean sim — Early: {early.mean():.4f}  Mid: {mid.mean():.4f}  Late: {late.mean():.4f}")

    # Friedman (if all 3 thirds available)
    if len(cols_present) == 3:
        stat_f, p_f = friedmanchisquare(early, mid, late)
        print(f"  Friedman χ² = {stat_f:.4f}  p = {p_f:.4f}")

    # Wilcoxon: early vs late
    if len(early) > 0 and len(late) > 0:
        n_common = min(len(early), len(late))
        stat_w, p_w = wilcoxon(early[:n_common], late[:n_common])
        direction = "late > early ✓" if late[:n_common].mean() > early[:n_common].mean() else "early > late ✗"
        print(f"  Wilcoxon (early vs late): W={stat_w:.1f}  p={p_w:.4f}  [{direction}]")

    thirds_summary[dom] = {
        'early_mean': early.mean() if len(early) else np.nan,
        'mid_mean':   mid.mean()   if len(mid)   else np.nan,
        'late_mean':  late.mean()  if len(late)  else np.nan,
        'n':          len(wide),
    }

# ── 4. T2 — Per-session Spearman ρ (position vs cosine sim) ──────────────────
print("\n" + "=" * 65)
print("T2: PER-SESSION SPEARMAN ρ (word position vs cosine similarity)")
print("=" * 65)

session_rhos = []

for (pid, dom), grp in pairs_df.groupby(['participant_id', 'domain']):
    if len(grp) < 5:
        continue
    rho, _ = spearmanr(grp['position'], grp['cosine_sim'])
    session_rhos.append({'participant_id': pid, 'domain': dom, 'rho': rho})

rho_df = pd.DataFrame(session_rhos)

print(f"\nTotal sessions with ρ computed: {len(rho_df)}")
print(f"Overall median ρ: {rho_df['rho'].median():.4f}")

print("\nPer-domain:")
for dom in DOMAINS:
    sub = rho_df[rho_df['domain'] == dom]['rho'].dropna()
    if len(sub) < 3:
        continue
    stat_w, p_w = wilcoxon(sub, alternative='greater')   # one-sided: ρ > 0
    print(f"  [{dom}]  n={len(sub)}  median ρ={sub.median():.4f}  "
          f"mean ρ={sub.mean():.4f}  Wilcoxon p={p_w:.4f} "
          f"{'✓' if p_w < 0.05 else '✗'}")

# Overall one-sample Wilcoxon
all_rhos = rho_df['rho'].dropna()
stat_all, p_all = wilcoxon(all_rhos, alternative='greater')
print(f"\nOverall (all domains pooled): median ρ={all_rhos.median():.4f}  "
      f"Wilcoxon p={p_all:.4f} {'✓' if p_all < 0.05 else '✗'}")

# ── 5. T3 — Permutation baseline ──────────────────────────────────────────────
print("\n" + "=" * 65)
print("T3: PERMUTATION BASELINE (shuffle word order within session)")
print("=" * 65)

N_PERM = 2000
observed_rhos = all_rhos.values
perm_median_rhos = []

for _ in range(N_PERM):
    perm_rhos = []
    for (pid, dom), grp in pairs_df.groupby(['participant_id', 'domain']):
        if len(grp) < 5:
            continue
        shuffled_pos = grp['position'].sample(frac=1).values
        r, _ = spearmanr(shuffled_pos, grp['cosine_sim'].values)
        perm_rhos.append(r)
    perm_median_rhos.append(np.median(perm_rhos))

perm_arr      = np.array(perm_median_rhos)
obs_median    = np.median(observed_rhos)
p_perm        = float(np.mean(perm_arr >= obs_median))

print(f"Observed median ρ:   {obs_median:.4f}")
print(f"Null median ρ (mean of permuted medians): {perm_arr.mean():.4f}")
print(f"Permutation p: {p_perm:.4f}  {'✓' if p_perm < 0.05 else '✗'}")

# ── 6. VISUALIZATIONS ─────────────────────────────────────────────────────────

# ── Fig 1 — Mean cosine similarity per third per domain ───────────────────────
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4.5 * len(DOMAINS), 5),
                         sharey=False)
if len(DOMAINS) == 1:
    axes = [axes]

segment_order  = ['early', 'mid', 'late']
segment_colors = [C_EARLY, C_MID, C_LATE]
segment_labels = ['Early\n(1st third)', 'Mid\n(2nd third)', 'Late\n(3rd third)']

for ax, dom in zip(axes, DOMAINS):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_color('#CCCCCC')

    sub = session_thirds[session_thirds['domain'] == dom]
    means = [sub[sub['segment'] == s]['mean_sim'].mean() for s in segment_order]
    sems  = [sub[sub['segment'] == s]['mean_sim'].sem()  for s in segment_order]

    bars = ax.bar(segment_labels, means, color=segment_colors,
                  alpha=0.82, edgecolor='white', linewidth=0.5, zorder=3)
    ax.errorbar(segment_labels, means, yerr=sems,
                fmt='none', color=TEXT, capsize=4, linewidth=1.5, zorder=4)

    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                m + 0.005, f'{m:.3f}',
                ha='center', fontsize=8.5, color=TEXT, fontweight='500')

    ax.set_title(dom, fontsize=11, fontweight='500',
                 color=DOMAIN_COLORS.get(dom, TEXT), pad=8)
    ax.set_ylabel('Mean cosine similarity', fontsize=9, color=MUTED)
    ax.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9, colors=MUTED)

fig.suptitle(
    'Fig 1 — Mean Cosine Similarity per Retrieval Third\n'
    'Hypothesis: late third > early third (clustering increases over time)',
    fontsize=11, fontweight='500', color=TEXT, y=1.02)
plt.tight_layout()
plt.savefig('fig1_thirds_similarity.png', bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: fig1_thirds_similarity.png")


# ── Fig 2 — Distribution of session-level ρ per domain ────────────────────────
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4.5 * len(DOMAINS), 5),
                         sharey=False)
if len(DOMAINS) == 1:
    axes = [axes]

for ax, dom in zip(axes, DOMAINS):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_color('#CCCCCC')

    sub  = rho_df[rho_df['domain'] == dom]['rho'].dropna()
    col  = DOMAIN_COLORS.get(dom, C_LATE)
    ax.hist(sub, bins=12, color=col, alpha=0.75,
            edgecolor='white', linewidth=0.5, zorder=3)
    ax.axvline(sub.median(), color=TEXT, lw=2, ls='--',
               label=f'Median ρ = {sub.median():.3f}')
    ax.axvline(0, color=C_NULL, lw=1.5, ls=':',
               label='ρ = 0 (no trend)')
    ax.set_title(dom, fontsize=11, fontweight='500',
                 color=col, pad=8)
    ax.set_xlabel('Spearman ρ (position vs similarity)', fontsize=9, color=MUTED)
    ax.set_ylabel('Number of sessions', fontsize=9, color=MUTED)
    ax.legend(fontsize=8, framealpha=0.85, edgecolor='#CCCCCC')
    ax.tick_params(labelsize=9, colors=MUTED)
    ax.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(
    'Fig 2 — Per-session Spearman ρ: word position vs cosine similarity\n'
    'Positive ρ = clustering increases over time',
    fontsize=11, fontweight='500', color=TEXT, y=1.02)
plt.tight_layout()
plt.savefig('fig2_session_rho_dist.png', bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: fig2_session_rho_dist.png")


# ── Fig 3 — Observed vs permutation null distribution ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color('#CCCCCC')

ax.hist(perm_arr, bins=40, color=C_NULL, alpha=0.75,
        edgecolor='white', linewidth=0.4,
        label='Null distribution (shuffled order)', zorder=3)
ax.axvline(obs_median, color=C_LATE, lw=2.5,
           label=f'Observed median ρ = {obs_median:.4f}', zorder=5)
ax.axvline(perm_arr.mean(), color=MUTED, lw=1.5, ls='--',
           label=f'Null mean = {perm_arr.mean():.4f}', zorder=4)

p_str = f'p < 0.001' if p_perm < 0.001 else f'p = {p_perm:.4f}'
ax.text(obs_median + 0.002, ax.get_ylim()[1] * 0.7,
        f'{p_str}', fontsize=10, color=C_LATE, fontweight='500')

ax.set_xlabel('Median Spearman ρ across sessions', fontsize=10, color=MUTED)
ax.set_ylabel('Frequency', fontsize=10, color=MUTED)
ax.tick_params(labelsize=9, colors=MUTED)
ax.legend(fontsize=9, framealpha=0.85, edgecolor='#CCCCCC')
ax.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

ax.set_title(
    f'Fig 3 — Permutation test (N={N_PERM} shuffles)\n'
    f'Observed median ρ vs null: {p_str}',
    fontsize=11, fontweight='500', color=TEXT, pad=10)
plt.tight_layout()
plt.savefig('fig3_permutation_null.png', bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: fig3_permutation_null.png")


# ── Fig 4 — Example session: similarity vs position (best illustration) ────────
# Pick session with highest ρ that has enough pairs
best = rho_df.sort_values('rho', ascending=False).iloc[0]
ex_pid, ex_dom = best['participant_id'], best['domain']
ex_pairs = pairs_df[
    (pairs_df['participant_id'] == ex_pid) &
    (pairs_df['domain'] == ex_dom)
].sort_values('position')

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color('#CCCCCC')

col = DOMAIN_COLORS.get(ex_dom, C_LATE)
ax.scatter(ex_pairs['position'], ex_pairs['cosine_sim'],
           color=col, s=70, alpha=0.8, edgecolors='white',
           linewidths=0.6, zorder=4)
ax.plot(ex_pairs['position'], ex_pairs['cosine_sim'],
        color=col, alpha=0.4, lw=1.2, zorder=3)

# add smoothed trend
from numpy.polynomial.polynomial import polyfit
x_fit = ex_pairs['position'].values
y_fit = ex_pairs['cosine_sim'].values
c     = polyfit(x_fit, y_fit, 1)
ax.plot(x_fit, c[0] + c[1] * x_fit,
        color=TEXT, lw=2, ls='--', alpha=0.7,
        label=f'Linear trend  (ρ = {best["rho"]:.3f})')

# segment shading
n_ex  = ex_pairs['n_words'].iloc[0]
third = n_ex // 3
ax.axvspan(1,         third,     alpha=0.08, color=C_EARLY, label='Early third')
ax.axvspan(third,     2 * third, alpha=0.08, color=C_MID,   label='Mid third')
ax.axvspan(2 * third, n_ex,      alpha=0.08, color=C_LATE,  label='Late third')

ax.set_xlabel('Word position in sequence', fontsize=10, color=MUTED)
ax.set_ylabel('Cosine similarity with previous word', fontsize=10, color=MUTED)
ax.tick_params(labelsize=9, colors=MUTED)
ax.legend(fontsize=9, framealpha=0.85, edgecolor='#CCCCCC', loc='upper left')
ax.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

ax.set_title(
    f'Fig 4 — Example session (participant {ex_pid}, {ex_dom})\n'
    f'Cosine similarity vs word position — ρ = {best["rho"]:.3f}',
    fontsize=11, fontweight='500', color=TEXT, pad=10)
plt.tight_layout()
plt.savefig('fig4_example_session.png', bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: fig4_example_session.png")


# ── 7. Final summary table ─────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"\nT1 — Mean cosine similarity per third:")
print(f"{'Domain':<14} {'Early':>8} {'Mid':>8} {'Late':>8} {'Direction':>16}")
print("-" * 55)
for dom, vals in thirds_summary.items():
    direction = "late > early ✓" if vals['late_mean'] > vals['early_mean'] else "early > late ✗"
    print(f"{dom:<14} {vals['early_mean']:>8.4f} {vals['mid_mean']:>8.4f} "
          f"{vals['late_mean']:>8.4f} {direction:>16}")

print(f"\nT2 — Session-level Spearman ρ (position vs cosine sim):")
print(f"{'Domain':<14} {'n':>5} {'Median ρ':>10} {'Mean ρ':>10}")
print("-" * 42)
for dom in DOMAINS:
    sub = rho_df[rho_df['domain'] == dom]['rho'].dropna()
    if len(sub) > 0:
        print(f"{dom:<14} {len(sub):>5} {sub.median():>10.4f} {sub.mean():>10.4f}")

print(f"\nT3 — Permutation test:")
print(f"  Observed median ρ: {obs_median:.4f}")
print(f"  Null mean ρ:       {perm_arr.mean():.4f}")
print(f"  p = {p_perm:.4f}  {'✓ hypothesis supported' if p_perm < 0.05 else '✗ not significant'}")
print("=" * 65)

H_new: RETRIEVAL BECOMES MORE SEMANTICALLY CLUSTERED OVER TIME
VFT rows: 687 | Participants: 27
Domains: ['animals', 'body-parts', 'foods', 'colours']

Loading LaBSE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 217 unique words...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings ready.

Total consecutive pairs: 580
             mean  count
segment                 
early    0.660400    198
late     0.597056    184
mid      0.598627    198

T1: MEAN COSINE SIMILARITY PER THIRD (early / mid / late)

[animals]  n sessions = 24
  Mean sim — Early: 0.7316  Mid: 0.6654  Late: 0.6630
  Friedman χ² = 11.0833  p = 0.0039
  Wilcoxon (early vs late): W=52.0  p=0.0039  [early > late ✗]

[body-parts]  n sessions = 20
  Mean sim — Early: 0.6360  Mid: 0.5938  Late: 0.5679
  Friedman χ² = 2.8000  p = 0.2466
  Wilcoxon (early vs late): W=49.0  p=0.0362  [early > late ✗]

[foods]  n sessions = 25
  Mean sim — Early: 0.5989  Mid: 0.5315  Late: 0.5253
  Friedman χ² = 10.3200  p = 0.0057
  Wilcoxon (early vs late): W=67.0  p=0.0088  [early > late ✗]

[colours]  n sessions = 4
  Mean sim — Early: 0.7419  Mid: 0.6369  Late: 0.6654
  Friedman χ² = 4.5000  p = 0.1054
  Wilcoxon (early vs late): W=2.0  p=0.3750  [early > late ✗]

T2: PER-SESSION SPEARMAN ρ (word position vs c

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 217 unique words...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Total sessions analysed: 80

── Per-Domain Results ─────────────────────────────────────────────────────
Domain          N sessions   Median ρ   % positive   Wilcoxon p
--------------------------------------------------------------
animals                 27      0.049        55.6%      0.1518 
body-parts              23      0.200        65.2%      0.0809 
foods                   26      0.224        69.2%      0.0140 *
colours                  4      0.080        50.0%      0.3125 
--------------------------------------------------------------
ALL DOMAINS             80      0.207        62.5%      0.0027

Running permutation test (2000 iterations)...

Observed median ρ (pooled) : 0.2071
Permutation null mean      : 0.0000
Permutation p-value        : 0.0000

Saved session results → H2_prototype_proximity_results.csv


In [1]:
# ============================================================
# H4 — Domain Modulates Retrieval Strategy
# Taxonomic domains (animals, body-parts) show stronger
# semantic clustering effects than thematic domains (foods, colours)
# ============================================================

import os
import gc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import fcluster, linkage  # replaced sklearn
import warnings
warnings.filterwarnings('ignore')

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # fixed typo in key name

# ── 1. Load datasets ───────────────────────────────────────────────────────────
vft_df  = pd.read_csv('/kaggle/working/final_normalised_dataset.csv')
spam_df = pd.read_csv('/kaggle/working/final_filtered_mapped_dataset.csv')
# VFT  columns : participant_id, domain, word_order, word, IRT, devnagri_word
# SpAM columns : participant_id, domain, word, x_norm, y_norm, devnagri_word

vft_df  = vft_df.dropna(subset=['devnagri_word']).copy()
spam_df = spam_df.dropna(subset=['devnagri_word']).copy()

DOMAINS      = ['animals', 'body-parts', 'foods', 'colours']
TAXONOMIC    = ['animals', 'body-parts']
THEMATIC     = ['foods', 'colours']

# ── 2. Load LaBSE and embed ────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/LaBSE')
model = model.half()   # fixed typo: was `modell`

all_words = pd.concat([
    vft_df['devnagri_word'],
    spam_df['devnagri_word']
]).dropna().unique().tolist()

print(f"Encoding {len(all_words)} unique words...")
emb_array   = model.encode(all_words, batch_size=64, show_progress_bar=True)
word_to_emb = {w: emb_array[i] for i, w in enumerate(all_words)}

# ── Free model from memory once embeddings are done ───────────────────────────
del model
del emb_array
gc.collect()
print("Model freed from memory")


# ── Clustering helpers (scipy — less memory overhead than sklearn) ─────────────
def get_labse_cluster_label(words, embeddings, k):
    if len(embeddings) < 2:
        return [0] * len(embeddings)
    k = max(2, min(k, len(embeddings) - 1))
    Z = linkage(embeddings, method='ward')
    labels = fcluster(Z, k, criterion='maxclust')
    return labels - 1   # zero-indexed to match sklearn behaviour

def get_spam_clusters(x_norm, y_norm, k):
    coords = np.column_stack([x_norm, y_norm])
    if len(coords) < 2:
        return np.zeros(len(coords), dtype=int)
    k = max(2, min(k, len(coords) - 1))
    Z = linkage(coords, method='ward')
    return fcluster(Z, k, criterion='maxclust') - 1


# ── Pre-compute ALL cluster labels BEFORE the main loops ──────────────────────
print("Pre-computing VFT cluster labels...")
vft_cluster_lookup  = {}   # (pid, domain, word) → cluster_label

for (pid, domain), sess in vft_df.groupby(['participant_id', 'domain']):
    sess        = sess.sort_values('word_order')
    words       = sess['devnagri_word'].tolist()
    valid_words = [w for w in words if w in word_to_emb]
    embs        = [word_to_emb[w] for w in valid_words]
    if len(embs) < 4:
        continue
    k      = max(2, int(np.sqrt(len(embs))))
    labels = get_labse_cluster_label(valid_words, np.array(embs), k)
    for w, lbl in zip(valid_words, labels):
        vft_cluster_lookup[(pid, domain, w)] = lbl

print("Pre-computing SpAM cluster labels...")
spam_cluster_lookup = {}   # (pid, domain, word) → cluster_label

for (pid, domain), sess in spam_df.groupby(['participant_id', 'domain']):
    xs    = sess['x_norm'].values
    ys    = sess['y_norm'].values
    words = sess['devnagri_word'].tolist()
    if len(words) < 4:
        continue
    k      = max(2, int(np.sqrt(len(words))))
    labels = get_spam_clusters(xs, ys, k)
    for w, lbl in zip(words, labels):
        spam_cluster_lookup[(pid, domain, w)] = lbl

gc.collect()
print("All cluster labels pre-computed.")


# ════════════════════════════════════════════════════════════════════════════════
# PART A — WC vs BC IRT gap per session
# ════════════════════════════════════════════════════════════════════════════════

session_gaps = []   # one row per (participant, domain): WC_mean_IRT, BC_mean_IRT, gap

for domain in DOMAINS:
    vft_sub = vft_df[vft_df['domain'] == domain]

    for pid, sess in vft_sub.groupby('participant_id'):
        sess  = sess.sort_values('word_order').copy()
        words = sess['devnagri_word'].tolist()
        irts  = sess['IRT'].tolist()

        if len(words) < 4:
            continue

        valid_idx = [i for i, w in enumerate(words)
                     if (pid, domain, w) in vft_cluster_lookup]

        if len(valid_idx) < 4:
            continue

        wc_irts, bc_irts = [], []
        for j in range(1, len(valid_idx)):
            i0, i1 = valid_idx[j-1], valid_idx[j]
            if i1 != i0 + 1:
                continue
            irt = irts[i1]
            if pd.isna(irt) or irt <= 0:
                continue
            lbl0 = vft_cluster_lookup[(pid, domain, words[i0])]
            lbl1 = vft_cluster_lookup[(pid, domain, words[i1])]
            if lbl0 == lbl1:
                wc_irts.append(irt)
            else:
                bc_irts.append(irt)

        if not wc_irts or not bc_irts:
            continue

        wc_mean = np.mean(wc_irts)
        bc_mean = np.mean(bc_irts)
        gap     = bc_mean - wc_mean

        session_gaps.append({
            'participant_id' : pid,
            'domain'         : domain,
            'wc_mean_irt'    : wc_mean,
            'bc_mean_irt'    : bc_mean,
            'gap'            : gap,
            'n_wc'           : len(wc_irts),
            'n_bc'           : len(bc_irts),
        })

    gc.collect()

gaps_df = pd.DataFrame(session_gaps)
print(f"\nSessions with valid WC/BC split: {len(gaps_df)}")


# ════════════════════════════════════════════════════════════════════════════════
# PART B — WC Spearman ρ (SpAM jump distance vs IRT) per session
# ════════════════════════════════════════════════════════════════════════════════

wc_spearman_rows = []

for domain in DOMAINS:
    spam_sub = spam_df[spam_df['domain'] == domain]
    vft_sub  = vft_df[vft_df['domain'] == domain]

    for pid, spam_sess in spam_sub.groupby('participant_id'):
        vft_sess = vft_sub[vft_sub['participant_id'] == pid].sort_values('word_order').copy()

        if len(vft_sess) < 4 or len(spam_sess) < 4:
            continue

        # Build word → (x, y) lookup from SpAM
        coord_lookup = {
            row['devnagri_word']: (row['x_norm'], row['y_norm'])
            for _, row in spam_sess.iterrows()
        }

        words_seq = vft_sess['devnagri_word'].tolist()
        irts_seq  = vft_sess['IRT'].tolist()

        wc_distances, wc_irts = [], []

        for j in range(1, len(words_seq)):
            w0, w1 = words_seq[j-1], words_seq[j]
            irt = irts_seq[j]

            if pd.isna(irt) or irt <= 0:
                continue
            if w0 not in coord_lookup or w1 not in coord_lookup:
                continue

            lbl0 = spam_cluster_lookup.get((pid, domain, w0))
            lbl1 = spam_cluster_lookup.get((pid, domain, w1))
            if lbl0 is None or lbl1 is None or lbl0 != lbl1:
                continue

            x0, y0 = coord_lookup[w0]
            x1, y1 = coord_lookup[w1]
            dist = euclidean([x0, y0], [x1, y1])

            wc_distances.append(dist)
            wc_irts.append(irt)

        if len(wc_distances) < 4:
            continue

        rho, pval = spearmanr(wc_distances, wc_irts)
        wc_spearman_rows.append({
            'participant_id' : pid,
            'domain'         : domain,
            'wc_rho'         : rho,
            'wc_pval'        : pval,
            'n_pairs'        : len(wc_distances),
        })

    gc.collect()

wc_rho_df = pd.DataFrame(wc_spearman_rows)
print(f"Sessions with valid WC SpAM ρ: {len(wc_rho_df)}")


# ════════════════════════════════════════════════════════════════════════════════
# PART C — Statistical Tests
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*65)
print("TEST 1 — Kruskal-Wallis on WC–BC IRT gap across domains")
print("═"*65)

domain_gap_vals = {d: gaps_df[gaps_df['domain'] == d]['gap'].dropna().values
                   for d in DOMAINS}

valid_domains = {d: v for d, v in domain_gap_vals.items() if len(v) >= 3}
kw_stat, kw_p = kruskal(*valid_domains.values())
print(f"H = {kw_stat:.3f},  p = {kw_p:.4f}  ({'*significant*' if kw_p < 0.05 else 'ns'})")

print("\nPer-domain median gap (ms):")
for d in DOMAINS:
    vals = domain_gap_vals.get(d, [])
    if len(vals):
        print(f"  {d:<15}  median gap = {np.median(vals):>8.1f} ms   n = {len(vals)}")

print("\nTaxonomic vs Thematic Mann-Whitney U (gap):")
tax_gaps = np.concatenate([domain_gap_vals.get(d, []) for d in TAXONOMIC])
the_gaps = np.concatenate([domain_gap_vals.get(d, []) for d in THEMATIC])
u_stat, u_p = mannwhitneyu(tax_gaps, the_gaps, alternative='greater')
print(f"  Taxonomic median = {np.median(tax_gaps):.1f} ms")
print(f"  Thematic  median = {np.median(the_gaps):.1f} ms")
print(f"  U = {u_stat:.1f},  p = {u_p:.4f}  ({'*significant*' if u_p < 0.05 else 'ns'})")


print("\n" + "═"*65)
print("TEST 2 — Kruskal-Wallis on WC SpAM ρ across domains")
print("═"*65)

domain_rho_vals = {d: wc_rho_df[wc_rho_df['domain'] == d]['wc_rho'].dropna().values
                   for d in DOMAINS}

valid_rho_domains = {d: v for d, v in domain_rho_vals.items() if len(v) >= 3}
kw_stat2, kw_p2 = kruskal(*valid_rho_domains.values())
print(f"H = {kw_stat2:.3f},  p = {kw_p2:.4f}  ({'*significant*' if kw_p2 < 0.05 else 'ns'})")

print("\nPer-domain median WC SpAM ρ:")
for d in DOMAINS:
    vals = domain_rho_vals.get(d, [])
    if len(vals):
        print(f"  {d:<15}  median ρ = {np.median(vals):>6.3f}   n = {len(vals)}")

print("\nTaxonomic vs Thematic Mann-Whitney U (WC SpAM ρ):")
tax_rhos = np.concatenate([domain_rho_vals.get(d, []) for d in TAXONOMIC])
the_rhos = np.concatenate([domain_rho_vals.get(d, []) for d in THEMATIC])
u_stat2, u_p2 = mannwhitneyu(tax_rhos, the_rhos, alternative='greater')
print(f"  Taxonomic median ρ = {np.median(tax_rhos):.3f}")
print(f"  Thematic  median ρ = {np.median(the_rhos):.3f}")
print(f"  U = {u_stat2:.1f},  p = {u_p2:.4f}  ({'*significant*' if u_p2 < 0.05 else 'ns'})")


# ════════════════════════════════════════════════════════════════════════════════
# PART D — Figures
# ════════════════════════════════════════════════════════════════════════════════

BG      = '#FFFFFF'
CARD    = '#F5F7FA'
TEXT    = '#1A1A2E'
MUTED   = '#555770'
TAX_C   = '#1565C0'
THE_C   = '#C62828'
DOMAIN_COLORS = {
    'animals'    : '#1565C0',
    'body-parts' : '#0288D1',
    'foods'      : '#C62828',
    'colours'    : '#E65100',
}

plt.rcParams.update({
    'font.family'     : 'DejaVu Sans',
    'axes.facecolor'  : CARD,
    'figure.facecolor': BG,
})

# ── Figure 1: WC vs BC mean IRT per domain (grouped bar) ─────────────────────
fig1, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)
ax.set_facecolor(CARD)
for sp in ax.spines.values(): sp.set_color('#CCCCCC')

x    = np.arange(len(DOMAINS))
w    = 0.3
wc_means = [np.mean(gaps_df[gaps_df['domain']==d]['wc_mean_irt']) for d in DOMAINS]
bc_means = [np.mean(gaps_df[gaps_df['domain']==d]['bc_mean_irt']) for d in DOMAINS]
gaps_med = [np.median(domain_gap_vals.get(d, [np.nan])) for d in DOMAINS]

bars_bc = ax.bar(x - w/2, bc_means, width=w, color=[DOMAIN_COLORS[d] for d in DOMAINS],
                 alpha=0.55, label='Between-cluster', zorder=3, edgecolor='none')
bars_wc = ax.bar(x + w/2, wc_means, width=w, color=[DOMAIN_COLORS[d] for d in DOMAINS],
                 alpha=0.95, label='Within-cluster',  zorder=3, edgecolor='none')

for i, (d, gap) in enumerate(zip(DOMAINS, gaps_med)):
    ymax = max(bc_means[i], wc_means[i])
    ax.annotate(f'Δ{gap:.0f}ms', xy=(x[i], ymax + 80),
                ha='center', fontsize=8.5, color=MUTED, fontweight='bold')

ax.axvline(1.5, color='#AAAAAA', linestyle='--', linewidth=1.2, zorder=2)
ax.text(0.5, 1, 'Taxonomic', ha='center', fontsize=9, color=TAX_C,
        transform=ax.get_xaxis_transform(), va='bottom')
ax.text(2.5, 1, 'Thematic', ha='center', fontsize=9, color=THE_C,
        transform=ax.get_xaxis_transform(), va='bottom')

ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in DOMAINS], fontsize=10, color=TEXT)
ax.set_ylabel('Mean IRT (ms)', fontsize=10, color=MUTED)
ax.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

p1 = mpatches.Patch(color='#555770', alpha=0.55, label='Between-cluster (BC)')
p2 = mpatches.Patch(color='#555770', alpha=0.95, label='Within-cluster (WC)')
ax.legend(handles=[p1, p2], fontsize=9, framealpha=0.7, labelcolor=TEXT)

ax.set_title('H4 — WC vs BC Mean IRT by Domain\n'
             f'Kruskal-Wallis on gap: H={kw_stat:.2f}, p={kw_p:.4f}  |  '
             f'Taxonomic vs Thematic: p={u_p:.4f}',
             fontsize=11, color=TEXT, pad=12)

plt.tight_layout()
plt.savefig('/kaggle/working/H4_fig1_IRT_gap_by_domain.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
print("\nSaved → H4_fig1_IRT_gap_by_domain.png")
plt.close()


# ── Figure 2: Boxplot of session-level gap per domain ────────────────────────
fig2, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=BG)

# Panel A: IRT gap boxplot
ax_a = axes[0]
ax_a.set_facecolor(CARD)
for sp in ax_a.spines.values(): sp.set_color('#CCCCCC')

box_data_gap = [domain_gap_vals.get(d, []) for d in DOMAINS]
bp = ax_a.boxplot(box_data_gap, patch_artist=True, notch=False,
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color=MUTED),
                  capprops=dict(color=MUTED),
                  flierprops=dict(marker='o', markersize=4, alpha=0.5))

for patch, d in zip(bp['boxes'], DOMAINS):
    patch.set_facecolor(DOMAIN_COLORS[d])
    patch.set_alpha(0.8)

ax_a.axhline(0, color='#AAAAAA', linestyle='--', linewidth=1)
ax_a.axvspan(0.5, 2.5, alpha=0.06, color=TAX_C, zorder=0)
ax_a.axvspan(2.5, 4.5, alpha=0.06, color=THE_C, zorder=0)
ax_a.set_xticks(range(1, 5))
ax_a.set_xticklabels([d.capitalize() for d in DOMAINS], fontsize=9.5, color=TEXT)
ax_a.set_ylabel('BC − WC IRT gap (ms)', fontsize=10, color=MUTED)
ax_a.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax_a.set_axisbelow(True)
ax_a.set_title('A  Session-level WC–BC IRT Gap', fontsize=11, color=TEXT, pad=8)
ax_a.text(1.5, ax_a.get_ylim()[0], 'Taxonomic', ha='center', fontsize=8.5,
          color=TAX_C, va='bottom', style='italic')
ax_a.text(3.5, ax_a.get_ylim()[0], 'Thematic', ha='center', fontsize=8.5,
          color=THE_C, va='bottom', style='italic')

# Panel B: WC SpAM ρ boxplot
ax_b = axes[1]
ax_b.set_facecolor(CARD)
for sp in ax_b.spines.values(): sp.set_color('#CCCCCC')

box_data_rho = [domain_rho_vals.get(d, []) for d in DOMAINS]
bp2 = ax_b.boxplot(box_data_rho, patch_artist=True, notch=False,
                   medianprops=dict(color='white', linewidth=2),
                   whiskerprops=dict(color=MUTED),
                   capprops=dict(color=MUTED),
                   flierprops=dict(marker='o', markersize=4, alpha=0.5))

for patch, d in zip(bp2['boxes'], DOMAINS):
    patch.set_facecolor(DOMAIN_COLORS[d])
    patch.set_alpha(0.8)

ax_b.axhline(0, color='#AAAAAA', linestyle='--', linewidth=1)
ax_b.axvspan(0.5, 2.5, alpha=0.06, color=TAX_C, zorder=0)
ax_b.axvspan(2.5, 4.5, alpha=0.06, color=THE_C, zorder=0)
ax_b.set_xticks(range(1, 5))
ax_b.set_xticklabels([d.capitalize() for d in DOMAINS], fontsize=9.5, color=TEXT)
ax_b.set_ylabel('WC SpAM ρ (distance vs IRT)', fontsize=10, color=MUTED)
ax_b.yaxis.grid(True, color='#DDDDDD', linewidth=0.6, zorder=0)
ax_b.set_axisbelow(True)
ax_b.set_title('B  Session-level WC SpAM ρ (distance vs IRT)', fontsize=11, color=TEXT, pad=8)
ax_b.text(1.5, ax_b.get_ylim()[0], 'Taxonomic', ha='center', fontsize=8.5,
          color=TAX_C, va='bottom', style='italic')
ax_b.text(3.5, ax_b.get_ylim()[0], 'Thematic', ha='center', fontsize=8.5,
          color=THE_C, va='bottom', style='italic')

fig2.suptitle('H4 — Domain Modulates Retrieval Strategy: Taxonomic vs Thematic',
              fontsize=12, color=TEXT, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/H4_fig2_boxplots.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
print("Saved → H4_fig2_boxplots.png")
plt.close()


# ── Save numeric results ───────────────────────────────────────────────────────
gaps_df.to_csv('/kaggle/working/H4_session_gaps.csv', index=False)
wc_rho_df.to_csv('/kaggle/working/H4_wc_spearman.csv', index=False)
print("Saved → H4_session_gaps.csv")
print("Saved → H4_wc_spearman.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 258 unique words...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Model freed from memory
Pre-computing VFT cluster labels...
Pre-computing SpAM cluster labels...
All cluster labels pre-computed.

Sessions with valid WC/BC split: 80
Sessions with valid WC SpAM ρ: 45

═════════════════════════════════════════════════════════════════
TEST 1 — Kruskal-Wallis on WC–BC IRT gap across domains
═════════════════════════════════════════════════════════════════
H = 1.665,  p = 0.6447  (ns)

Per-domain median gap (ms):
  animals          median gap =    560.7 ms   n = 27
  body-parts       median gap =    540.1 ms   n = 23
  foods            median gap =     76.0 ms   n = 26
  colours          median gap =    895.4 ms   n = 4

Taxonomic vs Thematic Mann-Whitney U (gap):
  Taxonomic median = 550.4 ms
  Thematic  median = 76.0 ms
  U = 800.0,  p = 0.3114  (ns)

═════════════════════════════════════════════════════════════════
TEST 2 — Kruskal-Wallis on WC SpAM ρ across domains
═════════════════════════════════════════════════════════════════
H = 2.683,  p = 0.261